In [14]:
import sys
import pandas as pd
from pathlib import Path
sys.path.append(str(Path().resolve().parent))

from config import DATASETS_DIR

In [15]:
df1 = pd.read_csv(DATASETS_DIR / "dataset_1.csv")
df2 = pd.read_csv(DATASETS_DIR / "dataset_2.csv")

df1.to_pickle(DATASETS_DIR / "dataset_1.pkl")
df2.to_pickle(DATASETS_DIR / "dataset_2.pkl")

In [16]:
common_cols = df1.columns.intersection(df2.columns)
df1 = df1[common_cols]
df2 = df2[common_cols]

In [17]:
df2['depth'].equals(df2['d_depth'])

True

In [18]:
df2 = df2.sort_values(['well_id', 'time']).reset_index(drop=True)
df2['d_depth'] = df2.groupby('well_id')['depth'].diff()
df2 = df2.dropna(subset=['d_depth']).copy()
df2 = df2[df2['d_depth'] > 0].copy()
df2['speed'] = df2['d_depth'] / df2['d_time']
df2['EI'] = (df2['pressure_axis'] * df2['pressure_rotation'] * df2['rotation']) / df2['speed']

In [19]:
df1['source'] = 'df1'
df2['source'] = 'df2'

common_cols = [c for c in df1.columns if c in df2.columns]
df = pd.concat([df1[common_cols], df2[common_cols]], ignore_index=True)

print(f'\n Объединённый датасет: {len(df)} строк, {df["well_id"].nunique()} скважин')


 Объединённый датасет: 437510 строк, 2071 скважин


In [20]:
df.columns

Index(['id', 'time', 'processing_time', 'lon', 'lat', 'height', 'depth',
       'course', 'angle', 'num_sattelites', 'moving_type', 'rotation',
       'rod_change', 'pressure_air', 'pressure_axis', 'pressure_rotation',
       'electric_rotation', 'electric_pressure_axis', 'voltage_rotation',
       'voltage_pressure_axis', 'time_stamp', 'sequence_id', 'block_well_id',
       'rig_id', 'well_id', 'drilling_speed', 'rtk_status', 'd_depth',
       'd_time', 'speed', 'n/p', 'EI', 'EI 0-1', 'EI 1-2', 'source'],
      dtype='object')

In [41]:
cols_to_drop = ['id', 'time', 'lon', 'lat', 'height', 'course', 'angle', 'num_sattelites', 'moving_type', 'rod_change', 'pressure_air',
                'electric_rotation', 'electric_pressure_axis', 'voltage_rotation', 'voltage_pressure_axis', 'time_stamp', 'sequence_id',
                'block_well_id', 'rig_id', 'drilling_speed', 'rtk_status', 'EI','EI 0-1', 'EI 1-2', 'source', 'n/p']
df_cleaned = df.drop(columns=cols_to_drop)
df_cleaned.dropna(inplace=True)

In [42]:
df_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
Index: 437082 entries, 1 to 437509
Data columns (total 9 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   processing_time    437082 non-null  object 
 1   depth              437082 non-null  float64
 2   rotation           437082 non-null  float64
 3   pressure_axis      437082 non-null  int64  
 4   pressure_rotation  437082 non-null  int64  
 5   well_id            437082 non-null  int64  
 6   d_depth            437082 non-null  float64
 7   d_time             437082 non-null  float64
 8   speed              437082 non-null  float64
dtypes: float64(5), int64(3), object(1)
memory usage: 33.3+ MB


In [ ]:
df_cleaned.to_feather(DATASETS_DIR / "united.feather")

ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.

In [44]:
df_cleaned

,processing_time,depth,rotation,pressure_axis,pressure_rotation,well_id,d_depth,d_time,speed
1,2025-10-07 22:18:14.495,4.6965,104.892,12781,13144,25512,0.1212,6.0,0.020200
2,2025-10-07 22:18:19.239,4.8783,101.430,18971,18936,25512,0.1818,5.0,0.036360
3,2025-10-07 22:18:24.208,5.0298,101.430,20153,18184,25512,0.1515,5.0,0.030300
4,2025-10-07 22:18:29.385,5.1813,102.714,20162,17445,25512,0.1515,5.0,0.030300
5,2025-10-07 22:18:58.036,5.3934,47.724,1233,10808,25512,0.2121,27.0,0.007856
...,...,...,...,...,...,...,...,...,...
437505,2025-10-07 22:14:55.384,3.7572,104.598,10071,13434,25512,0.1818,5.0,0.036360
437506,2025-10-07 22:14:59.667,3.9390,105.978,12308,12992,25512,0.1818,5.0,0.036360
437507,2025-10-07 22:15:05.094,4.0299,105.978,12363,15220,25512,0.0909,6.0,0.015150
437508,2025-10-07 22:15:10.387,4.0905,102.420,13499,14416,25512,0.0606,5.0,0.012120
